# Assignment A2: Let HGQ choose the bit widths
**AIMS Edge ML for Science · Out: Friday 27 November · Due: Monday 30 November, 22:00 · 20% of the course grade · ~6 hours**

## Why this assignment
In Lab 2 you chose the bit widths yourself: you picked some numbers, trained, compared, and changed the numbers again. This works for a small network, but it is slow,
and it becomes impossible for a large network: our network has 4,256 weights, and a real trigger network has many more.

**High-Granularity Quantization (HGQ)**, from the Day 3 lecture, chooses the bit widths automatically. Every weight and every input gets its *own* bit width, and these
bit widths are learned by gradient descent, at the same time as the weights. If a bit width becomes **0**, that weight (or that input) is removed completely.
One number, **β**, controls the balance: a larger β puts more weight on reducing hardware cost, and less on physics performance.

## Why a W tagger
In Lab 2 (Part A3) you saw that when you reduce the number of bits, the **W and Z jets lose accuracy first**, and most of the new mistakes are W jets classified as Z jets
and Z jets classified as W jets. The top tagger is affected less. W and Z bosons both decay to two quarks, and their masses differ by only about 11 GeV (80 vs. 91 GeV),
so the network must keep small differences between jets, and those small differences are the first thing lost when bits are removed.
So the **W tagger** is the place where a careful choice of bit widths matters most, and it is the tagger you will optimise here.

**Physics metric:** the **W-tag background rejection at 50% W efficiency** = 1 / (fraction of non-W jets that pass, when half of the W jets pass). Higher is better.
**Hardware target:** about **50k BOPs**. This is an approximate target: get close to it, and report how close you are.

## BOPs and EBOPs
You know **BOPs** from Day 3: for every multiplication, (number of bits of one operand) × (number of bits of the other operand), summed over all multiplications.
For a dense layer where all weights have the same bit width b_w and all inputs have the same bit width b_in, BOPs = (number of multiplications) × b_w × b_in.

HGQ reports **EBOPs** ("effective BOPs"). It is the same quantity, but counted for each connection separately, with the bit widths that HGQ has learned:
for the weight that connects input *i* to neuron *j*, it adds (bits of that weight) × (bits of input *i*). A weight with 0 bits, or an input with 0 bits, adds nothing,
because that multiplication is not built at all. If every weight and every input in a layer has the same bit width, EBOPs and BOPs are the same
(HGQ's count differs from the simple sum by about 1%, because of small details in how it counts). So you can put HGQ models and Lab 2-style models on the same plot.

## What you do
| Part | | Weight | Question |
|---|---|---|---|
| 1. Worked example | 🟩 | 5% | Train one HGQ model: 5 epochs with β = 0, then β increases slowly. Plot W rejection against EBOPs after every epoch. |
| 2. Choosing and fine-tuning a model | 🟩 code · 🟥 answers | 25% | From each training, take the model just below 100k, 50k and 10k EBOPs, freeze its bit widths and train it for 5 more epochs. 3 seeds. How much does the extra training help? |
| 3. HGQ vs. uniform bit widths | 🟩 code · 🟥 answers | 20% | Train Lab 2-style Brevitas models with the same bit width everywhere, 3 seeds each. At which costs is HGQ better, and is the difference larger than the seed-to-seed spread? |
| 4. What HGQ removes | 🟩 code · 🟥 answers | 25% | At 100k, 50k, 10k and 1k EBOPs: which jet features are still used, how many neurons are left in each layer, and how many bits remain? |
| 5. Your own improvement | 🟩 code · 🟥 answers | 15% | Try one idea of your own to get a better W tagger at about 50k (3 seeds). Report the result, also if it did not help. |
| Fair comparisons and clear plots | | 10% | Same training data and number of epochs within each comparison; 3 seeds; readable plots; complete PDF. |

🟩 code: use any tools you like, but you must be able to explain every line. 🟥 answers: write them yourself, without AI tools, and support every statement with a number
from your plots or tables.

**How to submit**
1. Run every cell, so that all outputs and plots are visible. Write your answers in the cells marked *Your answer*.
2. **File → Print → Save as PDF.** Check that your plots and printed numbers appear in the PDF.
3. Name the file `A2_Surname_Firstname.pdf`.
4. Email it **by Monday 30 November, 22:00** to **aymantarig@aims.ac.za** and **hassanh@aims.ac.za**, with **sagar@slac.stanford.edu** in CC.
   Subject: `[EdgeML] A2 – Surname Firstname`.

**Compute.** Use **Runtime → Change runtime type → T4 GPU.** One HGQ training (65 epochs) takes about 9 minutes, and each fine-tuning (5 epochs) under a minute.
**The whole assignment needs about 1.5 hours of training. Start early: do not leave it for the last evening.** Most of the training runs in loops that do not need you. Every result is saved to your Google Drive as soon as it is computed, so if Colab disconnects you lose nothing: run the setup cells again and continue.

## 0. Setup

In [ ]:
# --- Colab setup: run once per session (~2 min: installs HGQ2 and Brevitas) ---
import os, sys, glob, subprocess
os.environ["KERAS_BACKEND"] = "torch"          # HGQ runs on Keras; we use Keras' PyTorch backend. Must be set before Keras is imported.
IN_COLAB = "google.colab" in sys.modules
LAB = "A2_compression"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    # quantizers<2: HGQ2 0.2 needs the 1.x version of this package (pip would otherwise pick 2.0, which HGQ2 can't import)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "HGQ2", "quantizers<2", "brevitas"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/edgeml_A2"
    os.chdir(f"{ROOT}/{LAB}")
else:
    SAVE_DIR = "my_results"
os.makedirs(SAVE_DIR, exist_ok=True)
print("working directory:", os.getcwd(), "| results saved in", SAVE_DIR)

In [ ]:
import time, warnings
warnings.filterwarnings("ignore")
sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, keras
from hgq.layers import QDense
from hgq.config import LayerConfigScope, QuantizerConfigScope
from hgq.utils import trace_minmax
from common.jets import load_jets, CLASSES
from common.compress import qat_mlp, train, bops, bkg_rejection

FAST = os.environ.get("FASTML_FAST") == "1"                            # instructor quick-test switch; leave it alone
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("keras backend:", keras.backend.backend(), "| device:", DEVICE)

X_train, X_test, y_train, y_test, FEATURES = load_jets()
idx = np.random.default_rng(0).permutation(len(X_train))[:200_000]    # the same 200k training jets for every model
Xtr, ytr = X_train[idx], y_train[idx]
Xte, yte = X_test[:50_000], y_test[:50_000]
W = CLASSES.index("w")
SIZES = (16, 64, 32, 32, 5)                                            # the Lab 2 network
WARMUP_EPOCHS = 1 if FAST else 5                                       # HGQ: epochs with β = 0 at the start, so the network learns before bits are removed
RAMP_EPOCHS = 6 if FAST else 60                                        # HGQ: epochs in which β rises from 1e-6 to 1e-3
FT_EPOCHS = 2 if FAST else 5                                           # HGQ: epochs of fine-tuning after a model is chosen
EPOCHS_BREVITAS = 4 if FAST else 40                                    # Brevitas: about the same total training as an HGQ model (fair)
RESULTS_FILE = f"{SAVE_DIR}/results.csv"
print("features:", list(FEATURES))

**Bookkeeping, scoring and the front.** Same idea as Lab 2: every model you score goes into `RESULTS` (a row with its name, kind, seed, cost, rejection and accuracy),
which is saved to your Drive after every model and reloaded when you come back. `summary()` averages each design over its seeds; `plot_front()` draws everything.

In [ ]:
def w_rejection(p, y=yte, eff=0.5):
    """W-tag background rejection at `eff` W efficiency, from class probabilities p (n, 5)."""
    s, b = p[y == W, W], p[y != W, W]
    fpr = (b >= np.quantile(s, 1 - eff)).mean()
    return float("inf") if fpr == 0 else float(1 / fpr)

def probs_hgq(model, X=Xte):
    return keras.ops.convert_to_numpy(keras.ops.softmax(model.predict(X, batch_size=8192, verbose=0)))

def probs_torch(model, X=Xte):
    with torch.no_grad():
        return torch.softmax(model.eval()(torch.from_numpy(X)), 1).numpy()

RESULTS = {}
if os.path.exists(RESULTS_FILE):
    RESULTS = {(r["name"], r["seed"]): r for r in pd.read_csv(RESULTS_FILE).to_dict("records")}
    print(f"reloaded {len(RESULTS)} results")

def record(name, kind, p, cost, seed=0, **extra):
    """p: test-set class probabilities. cost: EBOPs (HGQ) or BOPs (Brevitas): the same axis."""
    rej, acc = w_rejection(p), float((p.argmax(1) == yte).mean())
    RESULTS[(name, seed)] = dict(name=name, kind=kind, seed=seed, cost=cost, w_rej=rej, acc=acc, **extra)
    pd.DataFrame(list(RESULTS.values())).to_csv(RESULTS_FILE, index=False)
    print(f"{name:<38} seed {seed} | W rejection {rej:6.1f} | accuracy {acc:.4f} | cost {cost/1e3:8.1f}k")

def summary(kinds=None):
    df = pd.DataFrame(list(RESULTS.values()))
    if kinds is not None:
        df = df[df.kind.isin(kinds)]
    g = df.groupby("name").agg(kind=("kind", "first"), cost=("cost", "mean"), w_rej=("w_rej", "mean"), w_rej_sd=("w_rej", "std"),
                               acc=("acc", "mean"), seeds=("seed", "nunique")).reset_index()
    return g.fillna({"w_rej_sd": 0.0}).sort_values("cost").round({"cost": 0, "w_rej": 1, "w_rej_sd": 1, "acc": 4}).reset_index(drop=True)

def plot_front(kinds=None, budget=50_000, title="W tagger: physics vs. hardware cost"):
    g = summary(kinds)
    plt.figure(figsize=(10, 6))
    for kind, d in g.groupby("kind"):
        plt.errorbar(d.cost, d.w_rej, yerr=d.w_rej_sd, fmt="o-" if "ramp" in kind else "o", ms=5, capsize=3, alpha=.8, label=kind)
    plt.axvline(budget, c="r", ls="--", label=f"target: {budget/1e3:.0f}k")
    plt.xscale("log"); plt.xlabel("cost: EBOPs (HGQ) or BOPs (Brevitas), log scale"); plt.ylabel("W-tag background rejection @ 50% efficiency")
    plt.title(title); plt.grid(alpha=.3, which="both"); plt.legend(fontsize=8); plt.show()
    return g

**The HGQ network.** The same 16 → 64 → 32 → 32 → 5 network as Lab 2, built from HGQ's `QDense` layers. The two `QuantizerConfigScope` lines set HGQ's standard
quantizers (weights: learned bits per weight; activations: learned bits per input); `LayerConfigScope` switches on the EBOPs term in the loss, with a starting β.
You don't need to change these settings.

In [ ]:
def build_hgq(sizes=SIZES, beta0=1e-6, seed=0):
    keras.utils.set_random_seed(seed)                      # <- this is how you change the seed for an HGQ model
    with QuantizerConfigScope(place="all", default_q_type="kbi", overflow_mode="SAT_SYM"), \
         QuantizerConfigScope(place="datalane", default_q_type="kif", overflow_mode="WRAP"), \
         LayerConfigScope(enable_ebops=True, beta0=beta0):
        layers = [keras.layers.InputLayer((sizes[0],))]
        layers += [QDense(n, activation="relu") for n in sizes[1:-1]] + [QDense(sizes[-1])]
        model = keras.Sequential(layers)
    model.compile(optimizer=keras.optimizers.Adam(3e-3), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
    return model

def ebops(model):
    """Total EBOPs: HGQ keeps a running count in every quantized layer."""
    return float(sum(float(keras.ops.convert_to_numpy(l.ebops)) for l in model.layers if getattr(l, "ebops", None) is not None))

def finalize(model):
    """After training: calibrate the activation bit widths on training data (HGQ's trace_minmax) and return the exact EBOPs."""
    return float(trace_minmax(model, Xtr[:50_000], batch_size=8192))

def beta_vars(model):
    """Every layer's β (the weight of the EBOPs term), so a callback can change it during training."""
    return [v for l in model.layers for v in l.variables if "beta" in (getattr(v, "path", "") + v.name).lower()]

def set_beta(model, value):
    for v in beta_vars(model):
        v.assign(keras.ops.cast(keras.ops.full(keras.ops.shape(v), value), v.dtype))

def get_state(model):
    """A copy of everything in the model: weights, bit widths and β. Small: a few kB."""
    return [keras.ops.convert_to_numpy(v).copy() for v in model.weights]

def set_state(model, state):
    for v, a in zip(model.weights, state):
        v.assign(a)

**The three tools of this assignment.** Read them once; every part uses them.
- `run_ramp(seed)` trains one HGQ model: first `WARMUP_EPOCHS` epochs with β = 0 (the network learns the task before any bits are removed), then `RAMP_EPOCHS` epochs
  in which β rises from 10⁻⁶ to 10⁻³ (equal steps on a log scale). After **every** epoch it keeps a checkpoint: the epoch, its EBOPs, its W rejection and a copy of the model.
- `pick_just_below(checkpoints, target)` returns the most expensive checkpoint whose EBOPs is at or below the target: the biggest model that still fits.
- `fine_tune(model, epochs)` **freezes every bit width** (so the cost cannot change any more), sets β = 0 (with the bits frozen, the EBOPs term has nothing left to do),
  and trains only the weights for some more epochs.

In [ ]:
def run_ramp(seed=0, sizes=SIZES, warmup=None, ramp=None, b0=1e-6, b1=1e-3, log_name=None):
    """Warm-up with β = 0, then β rising from b0 to b1. Returns (model, checkpoints). If log_name is given, every epoch is also recorded in RESULTS."""
    warmup = WARMUP_EPOCHS if warmup is None else warmup
    ramp = RAMP_EPOCHS if ramp is None else ramp
    model = build_hgq(sizes=sizes, seed=seed)
    ckpts = []
    class Schedule(keras.callbacks.Callback):
        def on_epoch_begin(self, epoch, logs=None):
            r = epoch - warmup
            self.beta = 0.0 if r < 0 else b0 * (b1 / b0) ** (r / max(ramp - 1, 1))
            set_beta(model, self.beta)
        def on_epoch_end(self, epoch, logs=None):
            p = probs_hgq(model)
            ckpts.append(dict(epoch=epoch, beta=self.beta, ebops=ebops(model), w_rej=w_rejection(p), state=get_state(model)))
            if log_name:
                record(f"{log_name} epoch {epoch:02d}", "HGQ: one point per epoch of the ramp", p, ckpts[-1]["ebops"], seed=seed, beta=self.beta)
    t0 = time.time()
    model.fit(Xtr, ytr, batch_size=1024, epochs=warmup + ramp, verbose=0, callbacks=[Schedule()])
    print(f"ramp (seed {seed}, sizes {sizes}): {(time.time() - t0) / 60:.1f} min")
    return model, ckpts

def pick_just_below(ckpts, target):
    fits = [c for c in ckpts if c["ebops"] <= target]
    if not fits:
        print(f"no checkpoint at or below {target:,.0f} EBOPs: using the cheapest one")
        return min(ckpts, key=lambda c: c["ebops"])
    return max(fits, key=lambda c: c["ebops"])

def fine_tune(model, epochs, lr=1e-3):
    for l in model.layers:
        if hasattr(l, "kq"):
            l.kq.trainable = False                             # weight bit widths: frozen
            l.iq.trainable = False                             # input bit widths: frozen
    set_beta(model, 0.0)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
    model.fit(Xtr, ytr, batch_size=1024, epochs=epochs, verbose=0)

---
## Part 1. The worked example: one β ramp (5%)
Train one HGQ model with `run_ramp`. It takes about 9 minutes. Every epoch is recorded, so you can see the whole path from an expensive model (~200k EBOPs) to a very
cheap one (a few hundred EBOPs). We keep this model and its checkpoints in `RAMPS[0]`: Part 2 uses them.

In [ ]:
RAMPS = {}                                    # seed -> (model, checkpoints). Lost if Colab disconnects: then run the ramp again.
RAMPS[0] = run_ramp(seed=0, log_name="ramp seed 0")

In [ ]:
g = plot_front(kinds=["HGQ: one point per epoch of the ramp"], title="Part 1: one β ramp (one point per epoch)")

**How to read this plot.** Each point is the network at the end of one epoch. The first few points (β = 0) are the most expensive. Then, as β grows, the points move
from expensive and good (top right) to cheap and worse (bottom left), and they pass the 50k target. But they do not lie on a smooth line: the rejection changes a lot from
one epoch to the next, and it can go up and then down again. The reason is that the loss has two terms that work against each other: the physics term wants more bits,
and the EBOPs term wants fewer bits. Every time β increases, the network has to find a new balance between them, and it has not finished doing so when the epoch ends.
So a checkpoint taken in the middle of the ramp is not a finished model. Part 2 shows how to turn one into a finished model.

---
## Part 2. Choosing and fine-tuning a model (25%)
To get a finished model at a chosen cost: take the checkpoint just below the target (`pick_just_below`), load it, and fine-tune it for 5 epochs with its bit widths
frozen (`fine_tune`). The cost stays where it was chosen, and the weights get time to adapt to the bits that are left. At the end, `finalize` calibrates the input ranges,
which can change the cost by a few percent. The function below does all of this for one target, and records the W rejection twice: right after choosing, and after fine-tuning.

**A note on which epoch we keep.** Here we simply keep the model after the 5th fine-tuning epoch. In a real project you would set aside a **validation set**, evaluate
the model on it after every epoch, and keep the epoch with the best validation result (and stop training when it no longer improves: "early stopping"). The final numbers
would then be quoted on a separate **test set** that was never used to make any choice. We skip this to keep the assignment short, so treat small differences with care.

Here is the complete example for **50k EBOPs, seed 0**, using the ramp from Part 1:

In [ ]:
FINAL = {}                                    # (target, seed, label) -> the finished model's state; Part 4 looks inside these

def choose_and_fine_tune(ramp_result, target, seed, label=""):
    model, ckpts = ramp_result
    ck = pick_just_below(ckpts, target)
    set_state(model, ck["state"])
    name = f"HGQ {target / 1000:g}k{label}"
    record(f"{name}, no fine-tuning", "HGQ: chosen, no fine-tuning", probs_hgq(model), ebops(model), seed=seed, epoch=ck["epoch"])
    fine_tune(model, FT_EPOCHS)
    cost = finalize(model)
    record(f"{name}, fine-tuned", "HGQ: chosen and fine-tuned", probs_hgq(model), cost, seed=seed)
    FINAL[(target, seed, label)] = get_state(model)

t0 = time.time()
choose_and_fine_tune(RAMPS[0], 50_000, seed=0)
print(f"({(time.time() - t0) / 60:.1f} min)")

**Your task.**
1. With the ramp of seed 0 (`RAMPS[0]`), do the same for **100k and 10k EBOPs**.
2. Run **two more ramps**, with seeds 1 and 2: `RAMPS[1] = run_ramp(seed=1)` and `RAMPS[2] = run_ramp(seed=2)`. The seed sets every random choice (the initial weights
   and the order of the training batches).
3. For each of them, choose and fine-tune at **100k, 50k and 10k**.

Use the same `seed` in `choose_and_fine_tune` as in the ramp. Results with the same name and a different seed are averaged by `summary()`, and the plot shows the spread
(± one standard deviation) as an error bar. This is 2 ramps and 8 fine-tunings, about 25 minutes: write it as one loop and start it.

**How to compare results that have 3 seeds:** for each design, compute the mean and the spread (standard deviation) over the 3 seeds (`summary()` does this for you).
Call two designs different only if their means differ by more than about **twice the larger of the two spreads**. Otherwise write "the same within the spread".
This is a correct and useful result, not a failure.

In [ ]:
# ✏️ Your code here: the remaining targets for seed 0, then ramps and targets for seeds 1 and 2.

In [ ]:
g = plot_front(kinds=["HGQ: chosen, no fine-tuning", "HGQ: chosen and fine-tuned"], title="Part 2: the chosen models before and after fine-tuning")
g

🟥 **Your answer (Part 2), 3–5 sentences, with numbers.**
(a) For each target (100k, 50k, 10k), give the W rejection (mean ± spread over 3 seeds) before and after fine-tuning.
(b) At which targets is the improvement from fine-tuning larger than the spread? At which is it not?
(c) How close is the final cost of each model to its target, after the calibration in `finalize`?

*Your answer:*

---
## Part 3. HGQ vs. uniform bit widths (20%)
Now the method from Lab 2: a Brevitas QAT network with **the same bit width for all weights and inputs**, trained for about the **same number of epochs** as an HGQ
model (40; a chosen HGQ model has had 5 warm-up epochs + its ramp epochs + 5 fine-tuning epochs), so that the comparison is fair. Here is one complete example with 4 bits. Its cost is ordinary BOPs (the same quantity as EBOPs, see the top of this notebook).
To change the seed, use `seed=` in `train_uniform` (it sets the initial weights and the order of the training batches).

In [ ]:
def train_uniform(bits, seed=0, epochs=EPOCHS_BREVITAS):
    torch.manual_seed(seed)
    m = qat_mlp([(bits, bits)] * 4)
    train(m, torch.from_numpy(Xtr), torch.from_numpy(ytr), epochs=epochs, bs=4096, seed=seed, device=DEVICE)
    return m

m4 = train_uniform(4, seed=0)
record("Brevitas uniform 4 bits", "Brevitas, same bits everywhere", probs_torch(m4), bops([(4, 4)] * 4), seed=0)

**Your task.** Uniform bits **8, 6, 4, 3, 2**, each with **3 seeds**, then one plot with these and your fine-tuned HGQ models from Part 2. (15 trainings, ~10 min.)

In [ ]:
# ✏️ Your loop over bit widths and seeds here.

In [ ]:
g = plot_front(kinds=["HGQ: chosen and fine-tuned", "Brevitas, same bits everywhere"], title="Part 3: HGQ vs. uniform bits")
g

🟥 **Your answer (Part 3), 3–5 sentences, with numbers.**
(a) Near 50k, how much higher is the W rejection of HGQ than that of the best uniform-bit model with a similar cost? Is the difference larger than the spread?
(b) Is there a range of costs where uniform bit widths are as good as HGQ (the same within the spread)?
(c) Find the HGQ model with about the same W rejection as uniform 6 bits. How many times smaller is its cost?

*Your answer:*

---
## Part 4. What HGQ removes (25%)
An HGQ layer stores the bits it learned, and you can read them directly. For any `QDense` layer `l` of a trained model `m` (e.g. `l = m.layers[0]` for the first layer):
```python
kb = keras.ops.convert_to_numpy(l.kq.bits)              # shape (inputs, outputs): the bits of every weight. 0 bits = that connection is never built
ib = keras.ops.convert_to_numpy(l.iq.bits).reshape(-1)  # shape (inputs,): the bits of every input to this layer. 0 bits = that input is gone
```
What a 0 means depends on the layer:
- **First layer:** the first layer only receives the inputs, and its inputs are the **16 jet features**. So if input *i* of the first layer has 0 bits (`ib[i] == 0`),
  or all of its weights have 0 bits (`(kb[i] == 0).all()`), the network **does not use the jet feature `FEATURES[i]` at all**. HGQ has removed that physics feature.
- **Later layers:** input *i* of layer *k* is the output of neuron *i* of layer *k − 1*. If that input has 0 bits, or all of its weights are 0, the **neuron is removed**:
  its output is never used.
- A single weight with `kb[i, j] == 0` is one removed connection (one multiplication that is not built).

The function below applies exactly these rules to every layer of a model. Read it, then use it.

In [ ]:
def what_survives(model):
    """For every layer: how many inputs (jet features or neurons) are still used, the fraction of connections that are built, and the mean bits of what remains."""
    rows = []
    for k, l in enumerate([l for l in model.layers if hasattr(l, "kq")]):
        kb = keras.ops.convert_to_numpy(l.kq.bits)
        ib = keras.ops.convert_to_numpy(l.iq.bits).reshape(-1)
        alive = (ib > 0) & (kb > 0).any(axis=1)                  # an input survives if it has bits AND at least one connection
        built = (kb > 0) & alive[:, None]                        # a connection is built if its weight has bits and its input survives
        rows.append(dict(layer=k, inputs=len(ib), inputs_kept=int(alive.sum()), connections_built=f"{built.mean():.0%}",
                         mean_weight_bits=round(float(kb[built].mean()), 2) if built.any() else 0.0,
                         mean_input_bits=round(float(ib[alive].mean()), 2) if alive.any() else 0.0))
        if k == 0:
            kept = [str(FEATURES[i]) for i in np.where(alive)[0]]
    return pd.DataFrame(rows), kept

def finished_model(target, seed, label="", sizes=SIZES):
    """Rebuild a finished model from Part 2 (or Part 5) so you can look inside it."""
    m = build_hgq(sizes=sizes, seed=seed)
    set_state(m, FINAL[(target, seed, label)])
    return m

table, kept = what_survives(finished_model(50_000, seed=0))
print("features the 50k model (seed 0) still uses:", kept)
table

**Your task.** First make one more finished model at **1k EBOPs** from the seed-0 ramp: `choose_and_fine_tune(RAMPS[0], 1_000, seed=0)` (1 minute).
Then build one table for the seed-0 models at **100k, 50k, 10k and 1k**: for each cost, the number of features kept, the neurons kept per hidden layer, and the mean bits.
List the surviving features at 10k and 1k. Finally, look at the **10k models of seeds 1 and 2** (`finished_model(10_000, seed=1)`): are the *same* features kept?

In [ ]:
# ✏️ Your models and tables here.

🟥 **Your answer (Part 4), with numbers from your table.**
(a) As the cost goes down from 100k to 1k EBOPs, what does HGQ reduce first: the number of bits of the remaining weights, the number of connections, or the number of
neurons and features? Give the numbers that show it.
(b) Which jet features are still used at 1k EBOPs? For each one, write one line on what it measures about a jet. Are these the features you would expect to be useful for
separating W jets from the other jets? Why?
(c) Do the three 10k models (seeds 0, 1, 2) use the same features? What does this mean for how much you can trust the feature list of a single training?
(d) In Lab 2, reducing the bits of the first layer's inputs hurt the most. Does HGQ also keep more bits on the first layer's inputs than on the other layers?

*Your answer:*

---
## Part 5. Your own improvement at about 50k EBOPs (15%)
Part 2 gave you HGQ's finished model at 50k EBOPs, with 3 seeds. Try **one idea** to get a higher W rejection at about the same cost, also with 3 seeds, and compare
the two with the rule from Part 2. Give your runs a `label` (e.g. `label=", wide"`) so their names are different. Some ideas (choose one, or use your own):
- **Start with a wider network:** `run_ramp(seed, sizes=(16, 128, 64, 64, 5))`, then `choose_and_fine_tune(..., 50_000, seed, label=", wide")`. HGQ then removes more.
- **A slower ramp:** `run_ramp(seed, ramp=90)`, so the network has more epochs to adapt while β rises.
- **Longer fine-tuning:** change `FT_EPOCHS` (e.g. to 20) before calling `choose_and_fine_tune`.

Report the result in every case. "It did not help: the same within the spread" is a correct answer if your numbers show it.

In [ ]:
# ✏️ Your Part 5 experiment here (3 seeds).

In [ ]:
g = plot_front(kinds=["HGQ: chosen and fine-tuned", "Brevitas, same bits everywhere"], title="Final: all finished designs")
g[(g.cost > 25_000) & (g.cost < 100_000)]

🟥 **Your answer (Part 5), 3–5 sentences.** Describe your idea and why you expected it to help. Give the result (mean ± spread over 3 seeds) next to Part 2's 50k design,
and say whether it is better using the rule from Part 2. Finally, state your best W tagger at about 50k: its network size, its EBOPs and its W rejection.

*Your answer:*